In [6]:
import pandas as pd
from sqlalchemy import text
from common.utils import print_json
from config import settings

In [7]:
from clients.openai_client import OpenAIClient
from clients.openai_requests import OpenAIParserRequest, OpenAIToolRequest

llm_client = OpenAIClient(openai_settings=settings.openai)

In [8]:
from typing import Optional
from pydantic import BaseModel, Field
from config import BookConstraints

class CompareBooksIntent(BaseModel):
    """The purpose of this node is idenitfy user's intent to compare books"""
    compare_criteria: list[str]
    reasoning: str = Field(..., desscription="provide a reasoning for this")
    
class RecommendBooksIntent(BaseModel):
    """The purpose of this node is indentify user's intent to find new books"""
    
    authors: Optional[list[str]] = Field(default=None, description="Authors to include.")
    
    categories: Optional[list[str]] = Field(default=None, description="List of categories or subgenres.")
    genre: Optional[str] = Field(default=None, description="Main genre of the book.")
    
    is_children: Optional[bool] = Field(default=None, description="If True, include only child-friendly books.")
    
    min_pages: Optional[int] = Field(default=None, description="Minimum number of pages")
    max_pages: Optional[int] = Field(default=None, description="Maximum number of pages")
    
    min_year: Optional[int] = Field(default=None, description="Minimum published year")
    max_year: Optional[int] = Field(default=None, description="Maximum published year")
    
    min_rating: Optional[float] = Field(default=None, description="Minimum average rating count (0.0 is the lowest)")
    max_rating: Optional[float] = Field(default=None, description= "Max average rating (5.0 if the highest)")
    rating_counts: Optional[int] = Field(default=None, description="The total rating counts")
    
    reasoning: str = Field(..., desscription="provide a reasoning for this")

    


In [9]:
INTENT_SYSTEM_PROMPT = (
    """ You are a intent indentifier for ta books recommendation system.
    understand the user's query and create intents for the user query
    """
)

In [10]:
from openai import pydantic_function_tool
from app.common.messages import UserMessage
def get_request(query):
    req = OpenAIToolRequest(
        prompt=INTENT_SYSTEM_PROMPT,
        tool_models = [
            CompareBooksIntent,
            RecommendBooksIntent
        ],
        messages=[UserMessage(content=query)]
    )
    return req

In [11]:
req = get_request("compare Dune to the Iliad based on rating and page numbers. Then compare Dune to To Kill a Mockingbird theme. Then recommend me some books similar to the first two books. 9780765")

assistant_msg = await llm_client.execute(req)
assistant_msg = assistant_msg.output

herere1


In [12]:
print_json(assistant_msg)

{
  "role": "assistant",
  "id": "chatcmpl-E4VIV9R0GMVHhNhHTEzIjOhzCVGPK",
  "content": null,
  "tool_calls": [
    {
      "id": "call_tJiU3hMcRoJsCA9Hr8kUHhXH",
      "function": {
        "arguments": "{\"compare_criteria\": [\"rating\", \"page numbers\"], \"reasoning\": \"Compare Dune and The Iliad based on their ratings and page numbers to understand their popularity and length differences.\"}",
        "name": "CompareBooksIntent",
        "parsed_arguments": {
          "compare_criteria": [
            "rating",
            "page numbers"
          ],
          "reasoning": "Compare Dune and The Iliad based on their ratings and page numbers to understand their popularity and length differences."
        }
      },
      "type": "function"
    },
    {
      "id": "call_9KEHIDwhb5juapr4SCOLRXBZ",
      "function": {
        "arguments": "{\"compare_criteria\": [\"theme\"], \"reasoning\": \"Compare Dune and To Kill a Mockingbird based on their themes to analyze the core messages 